In [63]:
import wntr
import numpy as np
import pandas as pd

# --- Create and configure model ---
wn = wntr.network.WaterNetworkModel('Net3.inp')

# Parameters
K = 6*3600   # lookback (6h)
T = 4*3600   # forecast horizon (4h)
minimum_pressure = 3.52
required_pressure = 14.06

# Population and average demand
# population = wntr.metrics.population(wn)
# AED = wntr.metrics.average_expected_demand(wn)
nzd_junct = AED[AED > 0].index  # non-zero demand junctions

# Hydraulic settings (set BEFORE running simulation)
wn.options.hydraulic.demand_model = 'PDD'
wn.options.hydraulic.minimum_pressure = minimum_pressure
wn.options.hydraulic.required_pressure = required_pressure
wn.options.time.hydraulic_timestep = 1800        # 30 min
wn.options.time.report_timestep = 1800
wn.options.time.duration = 48*3600               # 48 hours

# --- Run baseline simulation ---
baseline_sim = wntr.sim.WNTRSimulator(wn)
baseline_results = baseline_sim.run_sim()

print(type(baseline_results.node['pressure']))
print(type(baseline_results.link['flowrate']))


# Extract and align results
baseline_pressure = baseline_results.node['pressure'][wn.junction_name_list]
baseline_flow = baseline_results.link['flowrate'][wn.pipe_name_list]
baseline_pressure.index = baseline_pressure.index.astype(int)
baseline_flow.index = baseline_flow.index.astype(int)

print("Pressure index type:", type(baseline_results.node['pressure'].index))
print("Pressure index sample:", baseline_results.node['pressure'].index[:10])

print("Flow index type:", type(baseline_results.link['flowrate'].index))
print("Flow index sample:", baseline_results.link['flowrate'].index[:10])


# Ensure no misalignment
assert baseline_pressure.index.equals(baseline_flow.index), "Time index mismatch"

# --- Compute baseline statistics ---
baseline_stats = {
    node_name: {
        'mean': baseline_pressure[node_name].mean(),
        'min': baseline_pressure[node_name].min(),
        'std': baseline_pressure[node_name].std()
    } for node_name in nzd_junct
}

print("Pressure columns:", baseline_results.node['pressure'].columns[:10])
print("Flow columns:", baseline_results.link['flowrate'].columns[:10])

baseline_stats_df = pd.DataFrame.from_dict(baseline_stats, orient='index')
print(baseline_stats_df.head())



# --- After you have baseline_pressure DataFrame ---
lookback_steps = K // wn.options.time.report_timestep  # e.g., 6h / 0.5h = 12 timesteps
rolling_mean = baseline_pressure.rolling(window=lookback_steps).mean()
rolling_min  = baseline_pressure.rolling(window=lookback_steps).min()
rolling_std  = baseline_pressure.rolling(window=lookback_steps).std()

forecast_steps = T // wn.options.time.report_timestep  # e.g., 4h / 0.5h = 8 timesteps
# leak_future = ... you get this from actual leak simulation, not baseline




<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
Pressure index type: <class 'pandas.core.indexes.base.Index'>
Pressure index sample: Index([0, 1800, 3600, 5400, 7200, 9000, 10800, 12600, 14400, 16200], dtype='int64')
Flow index type: <class 'pandas.core.indexes.base.Index'>
Flow index sample: Index([0, 1800, 3600, 5400, 7200, 9000, 10800, 12600, 14400, 16200], dtype='int64')
Pressure columns: Index(['10', '15', '20', '35', '40', '50', '60', '601', '61', '101'], dtype='object')
Flow columns: Index(['20', '40', '50', '60', '101', '103', '105', '107', '109', '111'], dtype='object')
          mean        min       std
15   35.771933  28.593627  3.519380
35   42.424683  40.612469  0.759422
101  38.132752  31.291711  4.646520
103  37.407080  30.981433  4.298366
105  40.436985  35.839900  2.865110


In [43]:
# --- Static node features ---
df_static_node = pd.DataFrame({
    'node': nzd_junct,
    'elevation': [wn.get_node(n).elevation for n in nzd_junct],
    # 'population': [population[n] for n in nzd_junct],
    'degree': [len(wn.get_links_for_node(n)) for n in nzd_junct]
})

# print('static node features')
# print(df_static_node.head())

# --- Static pipe features ---
df_static_pipe = pd.DataFrame([
    {
        'pipe': p,
        'length': wn.get_link(p).length,
        'diameter': wn.get_link(p).diameter,
        'age': getattr(wn.get_link(p), 'install_year', np.nan)
    } for p in wn.pipe_name_list
])

# --- Aggregate pipe features per node ---
pipe_features_per_node = []

for node in nzd_junct:
    links = wn.get_links_for_node(node)  # get all links (pipes) connected to the node
    pipe_lengths = [wn.get_link(l).length for l in links]
    pipe_diams   = [wn.get_link(l).diameter for l in links]
    pipe_ages    = [getattr(wn.get_link(l), 'install_year', np.nan) for l in links]

    # If ages are NaN, replace with random values (10-50 years)
    pipe_ages = [a if not np.isnan(a) else np.random.randint(10,51) for a in pipe_ages]

    pipe_features_per_node.append({
        'node': node,
        'pipe_length_avg': np.mean(pipe_lengths),
        'pipe_length_max': np.max(pipe_lengths),
        'pipe_diameter_avg': np.mean(pipe_diams),
        'pipe_diameter_max': np.max(pipe_diams),
        'pipe_age_avg': np.mean(pipe_ages)
    })

# print('aggregated pipe features')
# df_pipe_features_node = pd.DataFrame(pipe_features_per_node)
# print(df_pipe_features_node.head())


num_pipes = len(df_static_pipe)
df_static_pipe['age'] = np.random.randint(10, 51, size=num_pipes)

# print('maybe this is jsut for the age inclusion??')
# print(df_static_pipe.head())


# --- Align with baseline stats ---
df_baseline_stats = pd.DataFrame.from_dict(baseline_stats, orient='index').reset_index()
df_baseline_stats.rename(columns={'index': 'node'}, inplace=True)

# print('baseline stats:mean min std, not aggregates')
# print(df_baseline_stats.head())

# Merge node static features with baseline statistics
# df_node_features = df_static_node.merge(df_baseline_stats, on='node', how='inner')
# print('static(elevation, degree)+baseline(mean min std)')
# print(df_node_features.head())


# --- Merge node static + pipe features + baseline stats ---
df_node_features = df_static_node.merge(df_baseline_stats, on='node', how='inner')
df_node_features = df_node_features.merge(df_pipe_features_node, on='node', how='inner')
print('final merged: static+baseline+aggregate')
print(df_node_features.head())


final merged: static+baseline+aggregate
  node  elevation  degree       mean        min       std  pipe_length_avg  \
0   15     9.7536       1  35.771933  28.593627  3.519380          502.920   
1   35     3.8100       1  42.424683  40.612469  0.759422            9.144   
2  101    12.8016       3  38.132752  31.291711  4.646520         1837.944   
3  103    13.1064       2  37.407080  30.981433  4.298366          806.196   
4  105     8.6868       3  40.436985  35.839900  2.865110          684.276   

   pipe_length_max  pipe_diameter_avg  pipe_diameter_max  pipe_age_avg  
0          502.920           0.203200             0.2032     49.000000  
1            9.144           0.609600             0.6096     47.000000  
2         4328.160           0.389467             0.4572     38.666667  
3         1200.912           0.406400             0.4064     45.000000  
4          830.580           0.304800             0.3048     35.666667  


In [61]:
# ---------------------------------------------
# 1. Rolling statistics (time × node)
# ---------------------------------------------
rolling_mean_reset = rolling_mean.reset_index().melt(
    id_vars='index', var_name='node', value_name='rolling_mean'
)
rolling_min_reset = rolling_min.reset_index().melt(
    id_vars='index', var_name='node', value_name='rolling_min'
)
rolling_std_reset = rolling_std.reset_index().melt(
    id_vars='index', var_name='node', value_name='rolling_std'
)

# Rename index → time
for df in [rolling_mean_reset, rolling_min_reset, rolling_std_reset]:
    df.rename(columns={'index': 'time'}, inplace=True)

# Merge rolling mean/min/std
rolling_features = (
    rolling_mean_reset
    .merge(rolling_min_reset, on=['time', 'node'])
    .merge(rolling_std_reset, on=['time', 'node'])
)

# Make sure node column has same dtype as in df_node_features
rolling_features['node'] = rolling_features['node'].astype(str)
df_node_features['node'] = df_node_features['node'].astype(str)

# -----------------------------
# FILTER: keep only nodes present in df_node_features
# -----------------------------
valid_nodes = set(df_node_features['node'])
rolling_features = rolling_features[rolling_features['node'].isin(valid_nodes)]

# Check unmatched nodes (should be empty now)
unmatched = set(rolling_features['node']) - valid_nodes
print("Unmatched nodes after filtering:", list(unmatched))

# ---------------------------------------------
# 2. Merge rolling features with static+baseline+pipe features
# ---------------------------------------------
features_full = rolling_features.merge(df_node_features, on='node', how='left')

# ---------------------------------------------
# 3. Target variable: future pressure after forecast horizon
# ---------------------------------------------
forecast_steps = T // wn.options.time.report_timestep  # 4h / 0.5h = 8 steps
target_pressure = baseline_pressure.shift(-forecast_steps)

target_reset = target_pressure.reset_index().melt(
    id_vars='index', var_name='node', value_name='target_pressure'
)
target_reset.rename(columns={'index': 'time'}, inplace=True)

# Merge target into features
features_full = features_full.merge(target_reset, on=['time', 'node'], how='left')



# Merge rolling features with static features
features_full = rolling_features.merge(df_node_features, on='node', how='left')

# Drop overall mean/min/std columns if they exist
for col in ['mean', 'min', 'std', 'target_pressure']:
    if col in features_full.columns:
        features_full.drop(columns=col, inplace=True)



features_full['leak'] = 0  # baseline simulation: no leaks



# ---------------------------------------------
# 4. Final check
# ---------------------------------------------




print(features_full.head(15))
print(features_full.columns)


Unmatched nodes after filtering: []
     time node  rolling_mean  rolling_min  rolling_std  elevation  degree  \
0       0   15           NaN          NaN          NaN     9.7536       1   
1    1800   15           NaN          NaN          NaN     9.7536       1   
2    3600   15           NaN          NaN          NaN     9.7536       1   
3    5400   15           NaN          NaN          NaN     9.7536       1   
4    7200   15           NaN          NaN          NaN     9.7536       1   
5    9000   15           NaN          NaN          NaN     9.7536       1   
6   10800   15           NaN          NaN          NaN     9.7536       1   
7   12600   15           NaN          NaN          NaN     9.7536       1   
8   14400   15           NaN          NaN          NaN     9.7536       1   
9   16200   15           NaN          NaN          NaN     9.7536       1   
10  18000   15           NaN          NaN          NaN     9.7536       1   
11  19800   15     30.906929    28.59362

In [27]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import copy
import wntr

# --- Setup and time options ---
wn = wntr.network.WaterNetworkModel('Net3.inp')
wn.options.time.duration = 48*3600
wn.options.time.hydraulic_timestep = 1800
wn.options.time.report_timestep = 1800
wn.options.hydraulic.required_pressure = 15
wn.options.hydraulic.minimum_pressure = 0

# --- Failure probabilities by pipe diameter ---
pipe_diameters = wn.query_link_attribute('diameter', np.less_equal, 0.9144, link_type=wntr.network.Pipe)
failure_probability = pipe_diameters / pipe_diameters.sum()

misaligned_nodes = set(wn_leak.junction_name_list) - set(baseline_pressure.columns)
print("Nodes in leak sim but missing in baseline:", misaligned_nodes)

# --- Leak simulations ---
results = {}
np.random.seed(67823)
all_features = []

for i in range(5):
    wn_leak = copy.deepcopy(wn)
    misaligned_nodes = set(wn_leak.junction_name_list) - set(baseline_pressure.columns)
    print("Nodes in leak sim but missing in baseline:", misaligned_nodes)

    
    N = np.random.randint(1, 6)
    pipes_to_fail = np.random.choice(failure_probability.index, N, replace=False, p=failure_probability.values)
    time_of_failure = np.round(np.random.uniform(1, 10), 2)
    duration_of_failure = np.round(np.random.uniform(12, 24), 2)

    leak_counter = 0
    for pipe_to_fail in pipes_to_fail:
        pipe = wn_leak.get_link(pipe_to_fail)
        leak_diameter = pipe.diameter * 0.1
        leak_area = np.pi * (leak_diameter / 2) ** 2

        if pipe.start_node.node_type == 'Junction':
            leak_node = pipe.start_node
        elif pipe.end_node.node_type == 'Junction':
            leak_node = pipe.end_node
        else:
            continue

        leak_node.add_leak(
            wn_leak,
            area=leak_area,
            start_time=time_of_failure * 3600,
            end_time=(time_of_failure + duration_of_failure) * 3600
        )

    wn_leak.options.hydraulic.demand_model = 'PDD'
    sim = wntr.sim.WNTRSimulator(wn_leak)
    print(f"Realization {i}: Pipe Breaks={pipes_to_fail}, Start={time_of_failure}, End={time_of_failure+duration_of_failure}")
    results[i] = sim.run_sim()

    # Standardize index before using it
    results[i].node['pressure'].index = results[i].node['pressure'].index.astype(int)
    results[i].link['flowrate'].index = results[i].link['flowrate'].index.astype(int)

    t0_values = np.arange(6*3600, 42*3600+1, 6*3600)
    valid_t0_values = [t for t in t0_values if t <= baseline_pressure.index.max()]
    K = 6*3600

    for t0 in valid_t0_values:
        time_index = results[i].node['pressure'].index
        sim_times = pd.Index(np.arange(t0 - K, t0 + 1, wn.options.time.report_timestep, dtype=int))
        valid_times = results[i].node['pressure'].index.intersection(sim_times)
        if valid_times.empty:
            continue

        for node_name in results[i].node['pressure'].columns:
            ps = results[i].node['pressure'].loc[valid_times, node_name]
            if ps.empty or ps.isna().all():
                continue
        
            if node_name in baseline_pressure.columns:
                baseline_times = baseline_pressure.index.intersection(valid_times)
                feature_max_drop = baseline_pressure[node_name].loc[baseline_times].min() - ps.min() if not baseline_times.empty else np.nan
            else:
                feature_max_drop = np.nan
        
            all_features.append({
                'realization': i, 't0': t0, 'node': node_name,
                'pressure_mean': ps.mean(),
                'pressure_min': ps.min(),
                'pressure_std': ps.std(),
                'pressure_slope': np.polyfit(np.arange(len(ps)), ps.values, 1)[0] if len(ps) > 1 else 0,
                'pressure_max_drop': feature_max_drop,
                'pressure_pct_below_threshold': (ps < wn.options.hydraulic.minimum_pressure).mean()
            })
        

        for pipe_name in wn_leak.pipe_name_list:
            fs = results[i].link['flowrate'].loc[valid_times, pipe_name]
            if fs.empty:
                continue

            if pipe_name in baseline_flow.columns:
                baseline_times = baseline_flow.index.intersection(valid_times)
                feature_max_flow_drop = baseline_flow[pipe_name].loc[baseline_times].min() - fs.min() if not baseline_times.empty else np.nan
            else:
                feature_max_flow_drop = np.nan

            all_features.append({
                'realization': i, 't0': t0, 'pipe': pipe_name,
                'flow_mean': fs.mean(),
                'flow_min': fs.min(),
                'flow_std': fs.std(),
                'flow_slope': np.polyfit(np.arange(len(fs)), fs.values, 1)[0] if len(fs) > 1 else 0,
                'flow_max_drop': feature_max_flow_drop,
                'leak_flag': int(pipe_name in pipes_to_fail)
            })

# Convert to DataFrame
df_dynamic = pd.DataFrame(all_features)
flow_columns = ['flow_mean', 'flow_min', 'flow_std', 'flow_slope', 'flow_max_drop']
df_dynamic = df_dynamic.dropna(subset=flow_columns)

print("\n=== DF_DYNAMIC: FIRST 10 ROWS ===")
print(df_dynamic.head(10))

print("\n=== COLUMN TYPES ===")
print(df_dynamic.dtypes)

print("\n=== NULL VALUES COUNT ===")
print(df_dynamic.isna().sum())

print("\n=== SUMMARY STATS ===")
print(df_dynamic.describe())

# --- Check for zero-only features ---
numeric_cols = df_dynamic.select_dtypes(include=[np.number]).columns
zero_counts = (df_dynamic[numeric_cols] == 0).sum()
print("\n=== ZERO COUNTS PER NUMERIC COLUMN ===")
print(zero_counts)

# --- Check percentage of NaN and zero values for sanity ---
nan_percent = (df_dynamic.isna().sum() / len(df_dynamic) * 100).round(2)
zero_percent = (zero_counts / len(df_dynamic) * 100).round(2)

summary_check = pd.DataFrame({
    'NaN_%': nan_percent,
    'Zero_%': zero_percent
}).sort_values(by='NaN_%', ascending=False)

print("\n=== NAN & ZERO PERCENTAGE SUMMARY ===")
print(summary_check)



Nodes in leak sim but missing in baseline: set()
Nodes in leak sim but missing in baseline: set()
Realization 0: Pipe Breaks=['137' '311'], Start=1.05, End=16.95
Nodes in leak sim but missing in baseline: set()
Realization 1: Pipe Breaks=['202'], Start=8.92, End=30.72
Nodes in leak sim but missing in baseline: set()
Realization 2: Pipe Breaks=['233' '175' '247' '231' '60'], Start=8.08, End=28.549999999999997
Nodes in leak sim but missing in baseline: set()
Realization 3: Pipe Breaks=['225' '116' '281'], Start=4.31, End=26.759999999999998
Nodes in leak sim but missing in baseline: set()
Realization 4: Pipe Breaks=['305' '275' '179' '175' '287'], Start=6.93, End=27.28

=== DF_DYNAMIC: FIRST 10 ROWS ===
     realization     t0 node  pressure_mean  pressure_min  pressure_std  \
97             0  21600  NaN            NaN           NaN           NaN   
98             0  21600  NaN            NaN           NaN           NaN   
99             0  21600  NaN            NaN           NaN        

In [ ]:
# next step is to merge the datasets to be ready for RF
# here, we are doing HISTORICAL+STATIC FEATURES


# --- Prepare static features DataFrames ---
# --- Check dimensions for verification ---
print("Dynamic features shape:", df_dynamic.shape)
print("Node static features shape:", df_static_node.shape)
print("Pipe static features shape:", df_static_pipe.shape)

# --- Merge dynamic features with static features ---

# Node features
df_nodes = df_dynamic[df_dynamic['node'].notna()].merge(df_static_node, on='node', how='left')

# Pipe features
df_pipes = df_dynamic[df_dynamic['pipe'].notna()].merge(df_static_pipe, on='pipe', how='left')

# --- Combine nodes and pipes back into a single DataFrame ---
df_features = pd.concat([df_nodes, df_pipes], axis=0, ignore_index=True)

# --- Inspect final merged DataFrame ---
print("Final merged features shape:", df_features.shape)
print(df_features.head())


## TH FINAL DF_FEATURES DATA FRAME HAS ALL THE HISTORICAL AND STATIC FEATURES FOR RF MODELING!!